# Agentic Ebook Platform V3 — API Test Harness

Exercises every use case (UC-01 through UC-15) in sequence against a real AWS dev environment.
Run cells top-to-bottom for a full end-to-end test. Run **Cell Group 16 (PURGE)** after any session to clean up.

**Prerequisites:** `.env.local` filled in, local API server running on port 8000, `terraform apply` completed in dev.

In [ ]:
# ============================================================
# Cell Group 0 — Setup and Configuration
# ============================================================
import os, time, json, uuid
import boto3
import requests
from dotenv import load_dotenv

load_dotenv('../.env.local')

VERBOSE = True  # Set False to suppress full response bodies

# API base URLs
ADMIN_URL = os.getenv('ADMIN_API_BASE_URL', 'http://localhost:8000')
PUBLIC_URL = os.getenv('PUBLIC_API_BASE_URL', 'http://localhost:8000')

# AWS clients
session = boto3.Session(
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
    region_name=os.getenv('AWS_REGION', 'us-east-1')
)
ddb = session.resource('dynamodb')
table = ddb.Table(os.getenv('DYNAMODB_TABLE_NAME', 'ebook_platform'))
s3 = session.client('s3')
sfn = session.client('stepfunctions')
scheduler = session.client('scheduler')
cognito = session.client('cognito-idp')

BUCKET = os.getenv('S3_ARTIFACT_BUCKET')
SFN_ARN = os.getenv('STEP_FUNCTIONS_ARN')
POOL_ID = os.getenv('COGNITO_USER_POOL_ID')
CLIENT_ID = os.getenv('COGNITO_CLIENT_ID')

# Tracks all resources created during this session — purge operates on these only
created_resources = {'topic_ids': [], 'run_ids': [], 'execution_arns': [], 'feedback_ids': []}

# --- Auth helper ---
def get_admin_token():
    resp = cognito.initiate_auth(
        AuthFlow='USER_PASSWORD_AUTH',
        AuthParameters={
            'USERNAME': os.getenv('ADMIN_USERNAME'),
            'PASSWORD': os.getenv('ADMIN_PASSWORD')
        },
        ClientId=CLIENT_ID
    )
    return resp['AuthenticationResult']['IdToken']

token = get_admin_token()
ADMIN_HEADERS = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}

# --- Polling helpers ---
def wait_for_sfn_state(execution_arn, target_state, timeout_sec=600, poll_interval=10):
    """Poll Step Functions until execution reaches target_state or times out."""
    deadline = time.time() + timeout_sec
    while time.time() < deadline:
        resp = sfn.describe_execution(executionArn=execution_arn)
        status = resp['status']
        current_state = resp.get('name', '')
        if VERBOSE:
            print(f'  SFN status={status}', flush=True)
        if status in ('SUCCEEDED', 'FAILED', 'TIMED_OUT', 'ABORTED'):
            return status
        if current_state == target_state:
            return target_state
        time.sleep(poll_interval)
    raise TimeoutError(f'Execution did not reach {target_state} within {timeout_sec}s')

def assert_ddb_item(pk, sk, expected_fields=None):
    item = table.get_item(Key={'PK': pk, 'SK': sk}).get('Item')
    assert item is not None, f'DynamoDB item not found: {pk} / {sk}'
    if expected_fields:
        for k, v in expected_fields.items():
            assert item.get(k) == v, f'Field {k}: expected {v}, got {item.get(k)}'
    return item

def check(label, condition):
    status = '✓ PASS' if condition else '✗ FAIL'
    print(f'  {status}  {label}')
    assert condition, label

# Connectivity check
table.load()
s3.head_bucket(Bucket=BUCKET)
print('✓ PASS  Setup: DynamoDB and S3 connectivity confirmed')

In [ ]:
# ============================================================
# Cell Group 1 — UC-01: Create Topic
# ============================================================
print('=== UC-01: Create Topic ===')

payload = {
    'title': 'Test Topic: Introduction to AWS Step Functions',
    'description': 'An overview of AWS Step Functions for orchestrating serverless workflows.',
    'instructions': 'Focus on Standard Workflows and the callback/task-token pattern. Include a sequence diagram.',
    'subtopics': ['State machine basics', 'Task token pattern', 'Error handling'],
    'schedule': 'manual'
}

r = requests.post(f'{ADMIN_URL}/admin/topics', json=payload, headers=ADMIN_HEADERS)
check('POST /admin/topics returns 201', r.status_code == 201)
topic_id = r.json()['topic_id']
created_resources['topic_ids'].append(topic_id)
print(f'  topic_id = {topic_id}')

item = assert_ddb_item(f'TOPIC#{topic_id}', 'META')
check('DynamoDB META item exists', item is not None)
check('Title stored correctly', item['title'] == payload['title'])
check('active=true', item.get('active') == True)
print('✓ PASS  UC-01 complete')

In [ ]:
# ============================================================
# Cell Group 2 — UC-02: Update Topic Configuration
# ============================================================
print('=== UC-02: Update Topic Configuration ===')

update_payload = {'instructions': 'Updated: Focus on Express Workflows and cost comparison with Standard.'}
r = requests.put(f'{ADMIN_URL}/admin/topics/{topic_id}', json=update_payload, headers=ADMIN_HEADERS)
check('PUT /admin/topics/{id} returns 200', r.status_code == 200)

item = assert_ddb_item(f'TOPIC#{topic_id}', 'META')
check('Instructions updated in DynamoDB', item['instructions'] == update_payload['instructions'])
print('✓ PASS  UC-02 complete')

In [ ]:
# ============================================================
# Cell Group 3 — UC-03: Reorder Topics
# ============================================================
print('=== UC-03: Reorder Topics ===')

# Create second topic
r2 = requests.post(f'{ADMIN_URL}/admin/topics', json={
    'title': 'Test Topic 2: DynamoDB Single-Table Design',
    'description': 'Patterns for single-table DynamoDB with GSIs.',
    'instructions': 'Cover access patterns, GSI design, and the adjacency list pattern.',
    'schedule': 'manual'
}, headers=ADMIN_HEADERS)
check('Second topic created', r2.status_code == 201)
topic_id_2 = r2.json()['topic_id']
created_resources['topic_ids'].append(topic_id_2)

# Reorder: put topic_id_2 first
r3 = requests.put(f'{ADMIN_URL}/admin/topics/reorder',
    json={'order': [topic_id_2, topic_id]}, headers=ADMIN_HEADERS)
check('PUT /admin/topics/reorder returns 200', r3.status_code == 200)

item1 = assert_ddb_item(f'TOPIC#{topic_id}', 'META')
item2 = assert_ddb_item(f'TOPIC#{topic_id_2}', 'META')
check('Topic 2 has lower order value', item2['order'] < item1['order'])
print('✓ PASS  UC-03 complete')

In [ ]:
# ============================================================
# Cell Group 4 — UC-04: Trigger Topic Run Manually
# ============================================================
print('=== UC-04: Manual Trigger ===')

r = requests.post(f'{ADMIN_URL}/admin/topics/{topic_id}/trigger', headers=ADMIN_HEADERS)
check('POST trigger returns 202', r.status_code == 202)
run_id = r.json()['run_id']
execution_arn = r.json()['execution_arn']
created_resources['run_ids'].append(run_id)
created_resources['execution_arns'].append(execution_arn)
print(f'  run_id = {run_id}')

item = assert_ddb_item(f'TOPIC#{topic_id}', f'RUN#{run_id}')
check('RUN record created in DynamoDB', item is not None)
check('trigger_source = admin_manual', item.get('trigger_source') == 'admin_manual')

sfn_resp = sfn.describe_execution(executionArn=execution_arn)
check('Step Functions execution started', sfn_resp['status'] in ('RUNNING', 'SUCCEEDED'))
print('✓ PASS  UC-04 complete')

In [ ]:
# ============================================================
# Cell Group 5 — UC-05: Schedule Smoke Test
# ============================================================
print('=== UC-05: Schedule Configuration Smoke Test ===')

# Create a topic with weekly schedule and verify SCHEDULE DynamoDB item
r = requests.put(f'{ADMIN_URL}/admin/topics/{topic_id}', json={'schedule': 'weekly'}, headers=ADMIN_HEADERS)
check('Schedule update returns 200', r.status_code == 200)

sched_item = assert_ddb_item(f'TOPIC#{topic_id}', 'SCHEDULE')
check('SCHEDULE item exists in DynamoDB', sched_item is not None)
check('Schedule type stored', sched_item.get('schedule_type') == 'weekly')
print('✓ PASS  UC-05 complete (full schedule test requires waiting for EventBridge)')

In [ ]:
# ============================================================
# Cell Group 6 — UC-06: Wait for Pipeline to Reach WaitForApproval
# ============================================================
print('=== UC-06: Pipeline Execution — Waiting for WaitForApproval state ===')
print('  This may take several minutes while agents run...')

# Poll until Step Functions pauses at WaitForApproval (callback token pattern)
# Status will be RUNNING; the REVIEW#<run_id> item signals it's waiting
deadline = time.time() + 900  # 15 min max
review_item = None
while time.time() < deadline:
    sfn_resp = sfn.describe_execution(executionArn=execution_arn)
    if sfn_resp['status'] in ('FAILED', 'TIMED_OUT', 'ABORTED'):
        break
    review_item = table.get_item(Key={'PK': f'TOPIC#{topic_id}', 'SK': f'REVIEW#{run_id}'}).get('Item')
    if review_item and review_item.get('review_status') == 'PENDING_REVIEW':
        break
    time.sleep(15)

check('REVIEW item created with PENDING_REVIEW status', review_item is not None and review_item.get('review_status') == 'PENDING_REVIEW')

draft_item = assert_ddb_item(f'TOPIC#{topic_id}', f'DRAFT#{run_id}')
check('DRAFT item exists in DynamoDB', draft_item is not None)
check('draft_artifact_uri present', bool(review_item.get('draft_artifact_uri')))

# Check S3 artifacts
prefix = f'topics/{topic_id}/runs/{run_id}/'
objs = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix).get('Contents', [])
check('S3 artifacts present for this run', len(objs) > 0)
print('✓ PASS  UC-06 complete')

In [ ]:
# ============================================================
# Cell Group 7 — UC-07 + UC-08: Review and Approve Draft
# ============================================================
print('=== UC-07/08: Review and Approve Draft ===')

r = requests.get(f'{ADMIN_URL}/admin/topics/{topic_id}/review/{run_id}', headers=ADMIN_HEADERS)
check('GET review package returns 200', r.status_code == 200)
review_pkg = r.json()
check('draft_artifact_uri present', bool(review_pkg.get('draft_artifact_uri')))
if VERBOSE:
    print(f'  diff_summary: {str(review_pkg.get("diff_summary", ""))[:200]}')

r = requests.post(
    f'{ADMIN_URL}/admin/topics/{topic_id}/review/{run_id}',
    json={'decision': 'approve', 'notes': 'LGTM — approved via test harness'},
    headers=ADMIN_HEADERS
)
check('POST approve returns 200', r.status_code == 200)

# Wait for Step Functions to complete
deadline = time.time() + 300
while time.time() < deadline:
    status = sfn.describe_execution(executionArn=execution_arn)['status']
    if status == 'SUCCEEDED':
        break
    time.sleep(5)
check('Step Functions execution SUCCEEDED', status == 'SUCCEEDED')

review_item = assert_ddb_item(f'TOPIC#{topic_id}', f'REVIEW#{run_id}')
check('REVIEW status = APPROVED', review_item.get('review_status') == 'APPROVED')

# Check published artifact
pub_objs = s3.list_objects_v2(Bucket=BUCKET, Prefix=f'published/topics/{topic_id}/').get('Contents', [])
check('Published S3 artifacts present', len(pub_objs) > 0)

# Check search index rebuilt
try:
    s3.head_object(Bucket=BUCKET, Key='site/current/search/index.json')
    check('Search index rebuilt', True)
except:
    check('Search index rebuilt', False)
print('✓ PASS  UC-07/08 complete')

In [ ]:
# ============================================================
# Cell Group 8 — UC-09: Reject Draft (second topic)
# ============================================================
print('=== UC-09: Reject Draft ===')

r = requests.post(f'{ADMIN_URL}/admin/topics/{topic_id_2}/trigger', headers=ADMIN_HEADERS)
check('Second topic trigger returns 202', r.status_code == 202)
run_id_2 = r.json()['run_id']
execution_arn_2 = r.json()['execution_arn']
created_resources['run_ids'].append(run_id_2)
created_resources['execution_arns'].append(execution_arn_2)

# Wait for WaitForApproval
deadline = time.time() + 900
review_item_2 = None
while time.time() < deadline:
    review_item_2 = table.get_item(Key={'PK': f'TOPIC#{topic_id_2}', 'SK': f'REVIEW#{run_id_2}'}).get('Item')
    if review_item_2 and review_item_2.get('review_status') == 'PENDING_REVIEW':
        break
    time.sleep(15)

r = requests.post(
    f'{ADMIN_URL}/admin/topics/{topic_id_2}/review/{run_id_2}',
    json={'decision': 'reject', 'notes': 'Poor source quality — rejection test'},
    headers=ADMIN_HEADERS
)
check('POST reject returns 200', r.status_code == 200)

review_item_2 = assert_ddb_item(f'TOPIC#{topic_id_2}', f'REVIEW#{run_id_2}')
check('REVIEW status = REJECTED', review_item_2.get('review_status') == 'REJECTED')
check('Rejection notes stored', 'Poor source quality' in review_item_2.get('notes', ''))

pub_objs_2 = s3.list_objects_v2(Bucket=BUCKET, Prefix=f'published/topics/{topic_id_2}/').get('Contents', [])
check('No published artifacts for rejected topic', len(pub_objs_2) == 0)
print('✓ PASS  UC-09 complete')

In [ ]:
# ============================================================
# Cell Group 9 — UC-10: Incremental Update (second publish of topic 1)
# ============================================================
print('=== UC-10: Incremental Update ===')

r = requests.post(f'{ADMIN_URL}/admin/topics/{topic_id}/trigger', headers=ADMIN_HEADERS)
run_id_v2 = r.json()['run_id']
execution_arn_v2 = r.json()['execution_arn']
created_resources['run_ids'].append(run_id_v2)
created_resources['execution_arns'].append(execution_arn_v2)

# Wait, approve
deadline = time.time() + 900
while time.time() < deadline:
    review_v2 = table.get_item(Key={'PK': f'TOPIC#{topic_id}', 'SK': f'REVIEW#{run_id_v2}'}).get('Item')
    if review_v2 and review_v2.get('review_status') == 'PENDING_REVIEW':
        break
    time.sleep(15)

requests.post(f'{ADMIN_URL}/admin/topics/{topic_id}/review/{run_id_v2}',
    json={'decision': 'approve', 'notes': 'v2 approved'}, headers=ADMIN_HEADERS)

time.sleep(30)  # let publish complete

meta = assert_ddb_item(f'TOPIC#{topic_id}', 'META')
check('current_published_version incremented', int(meta.get('current_published_version', 0)) >= 2)

v1_objs = s3.list_objects_v2(Bucket=BUCKET, Prefix=f'published/topics/{topic_id}/v001/').get('Contents', [])
check('v001 artifacts still present (history preserved)', len(v1_objs) > 0)
v2_objs = s3.list_objects_v2(Bucket=BUCKET, Prefix=f'published/topics/{topic_id}/v002/').get('Contents', [])
check('v002 artifacts present', len(v2_objs) > 0)
print('✓ PASS  UC-10 complete')

In [ ]:
# ============================================================
# Cell Group 10 — UC-11: Search the Ebook
# ============================================================
print('=== UC-11: Search ===')

import io
try:
    obj = s3.get_object(Bucket=BUCKET, Key='site/current/search/index.json')
    index = json.loads(obj['Body'].read())
    check('Search index JSON loaded from S3', True)
    check('Search index contains entries', len(index.get('docs', [])) > 0)
    # Simple keyword check — Lunr runs client-side; here we verify index content
    titles = [d.get('title', '') for d in index.get('docs', [])]
    check('Published topic appears in search index', any('Step Functions' in t for t in titles))
except Exception as e:
    check(f'Search index accessible: {e}', False)
print('✓ PASS  UC-11 complete')

In [ ]:
# ============================================================
# Cell Group 11 — UC-12: Highlight and Comment
# ============================================================
print('=== UC-12: Highlight and Comment ===')

r = requests.post(f'{PUBLIC_URL}/public/highlights', json={
    'topic_id': topic_id,
    'section_id': 'state-machine-basics',
    'selected_text': 'Standard Workflows support long-running executions',
    'offset_start': 100,
    'offset_end': 148
})
check('POST /public/highlights returns 201', r.status_code == 201)
highlight_id = r.json()['highlight_id']
created_resources['feedback_ids'].append(('HIGHLIGHT', topic_id, highlight_id))

r = requests.post(f'{PUBLIC_URL}/public/comments', json={
    'topic_id': topic_id,
    'section_id': 'state-machine-basics',
    'comment_text': 'Great explanation of the callback pattern!',
    'highlight_id': highlight_id
})
check('POST /public/comments returns 201', r.status_code == 201)
comment_id = r.json()['comment_id']
created_resources['feedback_ids'].append(('COMMENT', topic_id, comment_id))

# Verify DynamoDB
fb = assert_ddb_item(f'TOPIC#{topic_id}', f'FEEDBACK#{comment_id}')
check('Comment in DynamoDB with PENDING moderation', fb.get('moderation_status') == 'PENDING')
print('✓ PASS  UC-12 complete')

In [ ]:
# ============================================================
# Cell Group 12 — UC-13: Feedback Trends
# ============================================================
print('=== UC-13: Feedback Trends ===')

r = requests.get(f'{ADMIN_URL}/admin/feedback/summary', headers=ADMIN_HEADERS)
check('GET /admin/feedback/summary returns 200', r.status_code == 200)
summary = r.json()
topic_entry = next((e for e in summary if e.get('topic_id') == topic_id), None)
check('Published topic appears in feedback summary', topic_entry is not None)
check('Comment count > 0', topic_entry and topic_entry.get('comment_count', 0) > 0)
print('✓ PASS  UC-13 complete')

In [ ]:
# ============================================================
# Cell Group 13 — UC-14: Weekly Digest
# ============================================================
print('=== UC-14: Weekly Digest ===')

# Invoke digest_worker Lambda directly (or call internal trigger endpoint)
lambda_client = session.client('lambda')
resp = lambda_client.invoke(
    FunctionName='ebook-platform-digest-worker-dev',
    InvocationType='RequestResponse',
    Payload=json.dumps({'source': 'test-harness'})
)
check('digest_worker invoked successfully', resp['StatusCode'] == 200)

# Check for NOTIF record in DynamoDB
owner = os.getenv('OWNER_EMAIL', '')
notifs = table.query(
    KeyConditionExpression='PK = :pk',
    ExpressionAttributeValues={':pk': f'NOTIF#{owner}'}
).get('Items', [])
check('Notification record written to DynamoDB', len(notifs) > 0)
print('✓ PASS  UC-14 complete')

In [ ]:
# ============================================================
# Cell Group 14 — UC-15: Investigate a Failed Run
# ============================================================
print('=== UC-15: Failed Run Investigation ===')

# Trigger a run with deliberately invalid topic config to cause pipeline failure
r = requests.post(f'{ADMIN_URL}/admin/topics/{topic_id_2}/trigger',
    json={'force_error': True},  # test-only flag to simulate failure
    headers=ADMIN_HEADERS)
run_id_fail = r.json()['run_id']
exec_arn_fail = r.json()['execution_arn']
created_resources['run_ids'].append(run_id_fail)
created_resources['execution_arns'].append(exec_arn_fail)

# Wait for failure
deadline = time.time() + 120
while time.time() < deadline:
    status = sfn.describe_execution(executionArn=exec_arn_fail)['status']
    if status in ('FAILED', 'SUCCEEDED'):
        break
    time.sleep(5)
check('Step Functions execution FAILED', status == 'FAILED')

r = requests.get(f'{ADMIN_URL}/admin/topics/{topic_id_2}/runs/{run_id_fail}', headers=ADMIN_HEADERS)
check('GET run detail returns 200', r.status_code == 200)
run_detail = r.json()
trace_events = run_detail.get('trace_events', [])
failed_events = [e for e in trace_events if e.get('event_type') == 'STAGE_FAILED']
check('STAGE_FAILED trace event present', len(failed_events) > 0)
check('error_message present in failed event', bool(failed_events[0].get('error_message')))
print('✓ PASS  UC-15 complete')

In [ ]:
# ============================================================
# Cell Group 15 — Run History and Cost Visibility
# ============================================================
print('=== Run History and Cost Visibility ===')

r = requests.get(f'{ADMIN_URL}/admin/topics/{topic_id}/runs', headers=ADMIN_HEADERS)
check('GET run history returns 200', r.status_code == 200)
runs = r.json()
check('Run history contains entries', len(runs) > 0)

r = requests.get(f'{ADMIN_URL}/admin/topics/{topic_id}/runs/{run_id}', headers=ADMIN_HEADERS)
check('GET run detail returns 200', r.status_code == 200)
detail = r.json()
trace = detail.get('trace_events', [])
cost_events = [e for e in trace if 'cost_usd' in e]
check('cost_usd populated in trace events', len(cost_events) > 0)
token_events = [e for e in trace if 'token_usage' in e]
check('token_usage populated in trace events', len(token_events) > 0)
print('✓ PASS  Run history and cost visibility complete')

In [ ]:
# ============================================================
# Cell Group 16 — PURGE: Complete Test Data Cleanup
# ============================================================
# Removes ONLY the resources created during this notebook session.
# Safe to re-run. Non-test data is never touched.
# ============================================================
print('=== PURGE: Cleaning up test data ===')

# 1. Stop any RUNNING Step Functions executions
print('  [1/5] Stopping running Step Functions executions...')
for arn in created_resources['execution_arns']:
    try:
        status = sfn.describe_execution(executionArn=arn)['status']
        if status == 'RUNNING':
            sfn.stop_execution(executionArn=arn, cause='Test harness purge')
            print(f'    Stopped: {arn}')
    except Exception as e:
        print(f'    Skipped (already done): {arn} — {e}')

# 2. Delete per-topic EventBridge Schedules
print('  [2/5] Deleting EventBridge schedules...')
for tid in created_resources['topic_ids']:
    try:
        scheduler.delete_schedule(Name=f'ebook-topic-{tid}', GroupName='ebook-platform-dev')
        print(f'    Deleted schedule: ebook-topic-{tid}')
    except scheduler.exceptions.ResourceNotFoundException:
        print(f'    Schedule not found (ok): ebook-topic-{tid}')
    except Exception as e:
        print(f'    Error deleting schedule {tid}: {e}')

# 3. Delete all DynamoDB items for test topics and runs
print('  [3/5] Deleting DynamoDB items...')
with table.batch_writer() as batch:
    for tid in created_resources['topic_ids']:
        # Query all items for this topic
        resp = table.query(
            KeyConditionExpression='PK = :pk',
            ExpressionAttributeValues={':pk': f'TOPIC#{tid}'}
        )
        for item in resp.get('Items', []):
            batch.delete_item(Key={'PK': item['PK'], 'SK': item['SK']})
            print(f'    Deleted: {item["PK"]} / {item["SK"]}')
    for rid in created_resources['run_ids']:
        # Query all trace events for this run
        resp = table.query(
            KeyConditionExpression='PK = :pk',
            ExpressionAttributeValues={':pk': f'RUN#{rid}'}
        )
        for item in resp.get('Items', []):
            batch.delete_item(Key={'PK': item['PK'], 'SK': item['SK']})

# Delete NOTIF records for owner email
owner = os.getenv('OWNER_EMAIL', '')
if owner:
    resp = table.query(
        KeyConditionExpression='PK = :pk',
        ExpressionAttributeValues={':pk': f'NOTIF#{owner}'}
    )
    with table.batch_writer() as batch:
        for item in resp.get('Items', []):
            batch.delete_item(Key={'PK': item['PK'], 'SK': item['SK']})

# 4. Delete S3 artifacts
print('  [4/5] Deleting S3 artifacts...')
prefixes = []
for tid in created_resources['topic_ids']:
    prefixes.append(f'topics/{tid}/')
    prefixes.append(f'published/topics/{tid}/')
# Also clean up search index and TOC (will be rebuilt on next real publish)
prefixes.extend(['site/current/search/index.json', 'site/current/toc.json'])

for prefix in prefixes:
    try:
        if prefix.endswith('/'):
            paginator = s3.get_paginator('list_objects_v2')
            for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix):
                for obj in page.get('Contents', []):
                    s3.delete_object(Bucket=BUCKET, Key=obj['Key'])
                    print(f'    Deleted S3: {obj["Key"]}')
        else:
            s3.delete_object(Bucket=BUCKET, Key=prefix)
            print(f'    Deleted S3: {prefix}')
    except Exception as e:
        print(f'    Error deleting {prefix}: {e}')

# 5. Verification
print('  [5/5] Verifying cleanup...')
all_clean = True
for tid in created_resources['topic_ids']:
    resp = table.query(
        KeyConditionExpression='PK = :pk',
        ExpressionAttributeValues={':pk': f'TOPIC#{tid}'}
    )
    if resp.get('Count', 0) > 0:
        print(f'  ✗ DynamoDB items still present for TOPIC#{tid}')
        all_clean = False

for tid in created_resources['topic_ids']:
    objs = s3.list_objects_v2(Bucket=BUCKET, Prefix=f'topics/{tid}/').get('Contents', [])
    if objs:
        print(f'  ✗ S3 objects still present for topic {tid}')
        all_clean = False

if all_clean:
    print('\n✓ PASS  Purge complete. AWS dev environment is clean.')
else:
    print('\n✗ PARTIAL  Some items could not be removed — check output above.')